In [7]:
# CELL 1 — data ---------------------------------------------------------------
import numpy as np, pandas as pd, itertools

CSV = "/Users/amin/Documents/4_Learning/Summer school/Digital Twin/data/given/appendix_c.csv"
K   = ["pv_potential","daylighting_potential","relative_compactness","fsi_performance"]
TH  = [0.70, 0.70, 0.75, 0.80]

d     = pd.read_csv(CSV).set_index("configuration")
keep  = ((d[K]*100).round() >= np.array(TH)*100).all(axis=1)   # Meeting-4 screen, in hundredths
Z     = d.loc[keep, K]                                          # 10 designs, named index
ideal = Z.max()                                                 # z^Id from slide 31

print("feasible:", list(Z.index));  print("ideal:", ideal.values)


feasible: ['C1', 'C6', 'C8', 'C11', 'C14', 'C15', 'C18', 'C21', 'C25', 'C28']
ideal: [0.83 0.8  0.9  0.94]


In [8]:
# CELL 2 — the formula from slide 31, and a check -----------------------------
def L(w, a):
    """w = 4 weights summing to 1.  Returns one score per design; SMALLER IS BETTER."""
    gap = w * (ideal - Z)                                    # w_j * |z_j - z^Id_j|
    return gap.max(axis=1) if a == np.inf else (gap**a).sum(axis=1)**(1/a)

w = np.array([.25, .25, .25, .25])                           # engineering profile
print("L at a=1 picks   :", L(w, 1).idxmin())                # -> C14
print("weighted sum picks:", (Z*w).sum(axis=1).idxmax())     # -> C14   (must match)


L at a=1 picks   : C6
weighted sum picks: C6


In [9]:
# CELL 3 — enumerate every candidate decision maker ---------------------------------
# WHAT THIS IS: the case states neither the weights w (Kadzinski slide 15 calls this
# "imprecise weights") nor the compensation level alpha. So we enumerate both instead
# of assuming a point. The object being enumerated is Koksalan's "weight set"
# (Part I, slide 14); sweeping alpha as well is our own addition, from slide 31.

ALPHAS = [1, 2, 3, 4, np.inf]                    # the five curves drawn on slide 31

def weight_grid(steps):
    """All weight vectors on a lattice of 1/steps, every weight strictly positive."""
    return [np.array(c)/steps for c in itertools.product(range(1, steps+1), repeat=4)
            if sum(c) == steps]

def build(W):
    return pd.DataFrame([{"alpha": a, **dict(zip(["w1","w2","w3","w4"], w)), **L(w, a).to_dict()}
                         for a in ALPHAS for w in W])

def possible(M):
    return sorted(M[Z.index].idxmin(axis=1).unique())

WEIGHTS = weight_grid(20)                        # step 0.05
M = build(WEIGHTS)

# RESOLUTION CHECK — the step size is a modelling choice, so test it rather than assume it.
# A grid too coarse drops valid designs with no error at all (step 0.10 loses C28).
fine = build(weight_grid(40))                    # step 0.025
if possible(M) != possible(fine):
    raise ValueError(f"grid too coarse: step 0.05 gives {possible(M)}, "
                     f"step 0.025 gives {possible(fine)} — refine WEIGHTS")

print(f"{len(WEIGHTS)} weight vectors x {len(ALPHAS)} alphas = {len(M)} models")
print(f"resolution check passed (step 0.05 agrees with step 0.025)")
M.head()


969 weight vectors x 5 alphas = 4845 models
resolution check passed (step 0.05 agrees with step 0.025)


,alpha,w1,w2,w3,w4,C1,C6,C8,C11,C14,C15,C18,C21,C25,C28
0,1.0,0.05,0.05,0.05,0.85,0.0595,0.0660,0.012,0.043,0.0915,0.028,0.052,0.0765,0.037,0.0840
1,1.0,0.05,0.05,0.10,0.80,0.0610,0.0635,0.019,0.045,0.0865,0.033,0.052,0.0775,0.042,0.0815
2,1.0,0.05,0.05,0.15,0.75,0.0625,0.0610,0.026,0.047,0.0815,0.038,0.052,0.0785,0.047,0.0790
3,1.0,0.05,0.05,0.20,0.70,0.0640,0.0585,0.033,0.049,0.0765,0.043,0.052,0.0795,0.052,0.0765
4,1.0,0.05,0.05,0.25,0.65,0.0655,0.0560,0.040,0.051,0.0715,0.048,0.052,0.0805,0.057,0.0740


In [10]:
# CELL 4 — apply your statements, see what survives ---------------------------------
PREFS = [("C6","C8"), ("C6","C14")]      # (preferred, less preferred)

def possible(m):
    return sorted(m[Z.index].idxmin(axis=1).unique())

alive = M
print(f"start        : {len(alive)} models -> {possible(alive)}")

for better, worse in PREFS:
    alive = alive[alive[better] <= alive[worse]]     # smaller L = preferred
    if alive.empty:
        raise ValueError(f"{better} > {worse} contradicts an earlier statement")
    print(f"{better} > {worse}   : {len(alive)} models -> {possible(alive)}")

print("\nruled out:", sorted(set(Z.index) - set(possible(alive))))


start        : 4845 models -> ['C1', 'C11', 'C14', 'C15', 'C18', 'C21', 'C25', 'C28', 'C6', 'C8']
C6 > C8   : 3200 models -> ['C1', 'C11', 'C14', 'C18', 'C21', 'C28', 'C6']
C6 > C14   : 2568 models -> ['C1', 'C11', 'C18', 'C21', 'C28', 'C6']

ruled out: ['C14', 'C15', 'C25', 'C8']
